# My SNOTEL Analysis (Great Salt Lake Basin)
#### NWIS Site: 10109000 (Logan River)
#### Water Year: 2026

## Imports

In [17]:
from pynhd import NLDI
import geopandas as gpd
import pandas as pd
from supporting_scripts import getData, SNOTEL_Analyzer, dataprocessing, mapping
import os, datetime, numpy as np, warnings
warnings.filterwarnings("ignore")

In [ ]:
nldi = NLDI()

usgs_gage_id = "10109000"   # Logan River
WY = 2026
stateab = "Ut"              # IMPORTANT (Ut, Id, Wy, etc.)

## Delineate upstream watershed for the gage (and save shapefile)

In [19]:
nldi = NLDI()

usgs_gage_id = "10109000"   # Logan River (your pick)
WY = 2026
stateab = "Ut"              # IMPORTANT (Ut, Id, Wy, etc.)

## Map Basin

In [20]:
mapping.basin_mapping(basin, site_feature)

## Load all SNOTEL stations GeoJSON and clip to the basin

In [21]:
all_stations_gdf = gpd.read_file(
    "https://raw.githubusercontent.com/egagli/snotel_ccss_stations/main/all_stations.geojson"
).set_index("code")

all_stations_gdf = all_stations_gdf[all_stations_gdf["csvData"] == True]

gdf_in_bbox = all_stations_gdf[all_stations_gdf.geometry.within(basin.geometry.iloc[0])].copy()
gdf_in_bbox.reset_index(drop=False, inplace=True)

gdf_in_bbox["beginDate"] = [
    datetime.datetime.strftime(gdf_in_bbox["beginDate"].iloc[i], "%Y-%m-%d")
    for i in range(len(gdf_in_bbox))
]
gdf_in_bbox["endDate"] = [
    datetime.datetime.strftime(gdf_in_bbox["endDate"].iloc[i], "%Y-%m-%d")
    for i in range(len(gdf_in_bbox))
]

gdf_in_bbox[["code","name","beginDate","endDate"]]

,code,name,beginDate,endDate
0,1115_UT_SNTL,Klondike Narrows,2009-10-01,2026-03-04
1,1013_UT_SNTL,Temple Fork,2001-11-05,2026-03-04
2,823_UT_SNTL,Tony Grove Lake,1978-10-01,2026-03-04
3,1113_UT_SNTL,Tony Grove RS,2009-10-01,2026-03-04


## Checking 3-4 sites

In [22]:
gdf_in_bbox.shape, gdf_in_bbox["code"].tolist()

((4, 14), ['1115_UT_SNTL', '1013_UT_SNTL', '823_UT_SNTL', '1113_UT_SNTL'])

## Map Stations

In [31]:
mapping.snotel_mapping(gdf_in_bbox, basin, site_feature)

## Steo 3 issues

In [33]:
OutputFolder = "files/SNOTEL_UT"
os.makedirs(OutputFolder, exist_ok=True)

def code_to_station_id(code_str):
    # "1115_UT_SNTL" -> "1115"
    return code_str.split("_")[0]

def build_wcc_url(station_id, stateab, start_date, end_date):
    # matches what you printed in your logs
    return (
        "https://wcc.sc.egov.usda.gov/reportGenerator/view_csv/"
        "customMultiTimeSeriesGroupByStationReport/daily/start_of_period/"
        f"{station_id}:{stateab}:SNTL%7Cid=%22%22%7Cname/"
        f"{start_date},{end_date}/WTEQ::value?fitToScreen=false"
    )

def download_snotel_csv(site_name, code_str, stateab, start_date, end_date, out_folder):
    station_id = code_to_station_id(code_str)
    url = build_wcc_url(station_id, stateab, start_date, end_date)
    print("downloading:", code_str, site_name)
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    out_fp = os.path.join(out_folder, f"df_{code_str}_{stateab}_SNTL.csv")
    with open(out_fp, "wb") as f:
        f.write(r.content)
    return out_fp

downloaded = []
for i in gdf_in_bbox.index:
    fp = download_snotel_csv(
        gdf_in_bbox.loc[i, "name"],
        gdf_in_bbox.loc[i, "code"],
        stateab,
        gdf_in_bbox.loc[i, "beginDate"],
        gdf_in_bbox.loc[i, "endDate"],
        OutputFolder
    )
    downloaded.append(fp)

len(downloaded), downloaded[:3]

downloading: 1115_UT_SNTL Klondike Narrows
downloading: 1013_UT_SNTL Temple Fork
downloading: 823_UT_SNTL Tony Grove Lake
downloading: 1113_UT_SNTL Tony Grove RS


(4,
 ['files/SNOTEL_UT/df_1115_UT_SNTL_Ut_SNTL.csv',
  'files/SNOTEL_UT/df_1013_UT_SNTL_Ut_SNTL.csv',
  'files/SNOTEL_UT/df_823_UT_SNTL_Ut_SNTL.csv'])

In [34]:
# Cell 9 (Step 4 clean ALL downloaded CSVs so dataprocessing works)
import glob

for fp in glob.glob(os.path.join(OutputFolder, f"df_*_{stateab}_SNTL.csv")):
    # ignore NRCS comment header lines
    df = pd.read_csv(fp, comment="#")

    # find SWE column (often includes site name + (in))
    swe_src = [c for c in df.columns if "Snow Water Equivalent" in c]
    if not swe_src:
        print("No SWE col found in:", fp)
        continue
    swe_src = swe_src[0]

    # standardize name expected downstream
    df = df.rename(columns={swe_src: "Snow Water Equivalent (m) Start of Day Values"})

    # force numeric + convert inches -> meters
    swe = pd.to_numeric(df["Snow Water Equivalent (m) Start of Day Values"], errors="coerce").astype("float64")
    df["Snow Water Equivalent (m) Start of Day Values"] = swe * 0.0254

    # date + water year
    df["Date"] = pd.to_datetime(df["Date"])
    df["Water_Year"] = df["Date"].dt.year + (df["Date"].dt.month >= 10).astype(int)

    df.to_csv(fp, index=False)

print("cleaned")

cleaned


## Issues start with building sitedict........